In [0]:
dbutils.widgets.text("file_name","","Input Filename")
dbutils.widgets.text("environment","dev","Input Environment")

In [0]:
file_name = dbutils.widgets.get("file_name")
env = dbutils.widgets.get("environment")

In [0]:
%run ./01.config

In [0]:
def get_schema(filename):
    schema_dict = {"orders":"order_id STRING, customer_id STRING, product_id STRING,order_date DATE,quantity INT,total_amount DOUBLE","products":"product_id STRING, product_name STRING, category STRING, brand STRING, price DOUBLE","customers":"customer_id STRING,first_name STRING, last_name STRING, email STRING, city STRING, state STRING"}
    if schema_dict[filename]:
        return schema_dict[filename]
    else:
        raise Exception(f"Schema not defined for file : {filename}")
                         

In [0]:
def read_file_stream(path,filename):
    from pyspark.sql.functions import current_timestamp
    read_stream = (spark.readStream.format("CloudFiles")
                   .option("CloudFiles.format","parquet")
                   .schema(get_schema(filename))
                   .load(f"{path}/{filename}/")
                   .withColumn("extract_time",current_timestamp())
                   )
    print("Reading Success !!")
    print("*********************")
    return read_stream

In [0]:
def write_file_stream(streaming_df,path,env,filename):
    write_stream = (streaming_df.writeStream.outputMode("append")
     .option("checkpointLocation",f"{path}/{filename}/")
     .queryName(f"{filename}_write_stream")
     .trigger(availableNow=True)
     .toTable(f"{env}_retail.bronze.{filename}"))
    write_stream.awaitTermination()
    print("Write Success")
    print("*********************")


    

In [0]:
conf = Config()

In [0]:
read_stream_df = read_file_stream(path=conf.data_zone,filename=file_name)
write_file_stream(read_stream_df,conf.checkpoint_zone,env,file_name)